In [1]:
import pandas as pd
import numpy as np
import json
import os
import ast

In [2]:
df = pd.read_csv("..\data\movies_clean.csv")
df.head()

,adult,backdrop_path,movie_id,original_language,original_title,overview,popularity,poster_path,release_date,title,video,vote_average,vote_count,genres,keywords,cast,crew,release_year,score,director
0,FALSE,/kQM7o3NIkruIZLoQ9E2XzZQ8Ujl.jpg,783461,Hindi,लूप लपेटा,"When her boyfriend loses a mobster's cash, Sav...",56.311,/onGdT8sYi89drvSJyEJnft97rOq.jpg,04-02-2022,Looop Lapeta,FALSE,6.2,54.0,"['Action', 'Comedy', 'Crime']","['remake', 'looop lapeta', 'saade saati']","['Taapsee Pannu', 'Tahir Raj Bhasin', 'Shreya ...","['Tom Tykwer', 'Mukesh Chhabra', 'Tanuj Garg']",2022.0,5.611747,Aakash Bhatia
1,FALSE,/7w06baRS9VPm5RYz8lawTCLiR4j.jpg,592508,Hindi,Sooryavanshi,"A fearless, faithful albeit slightly forgetful...",45.077,/8p3mhjyLjHKtaAv8tFKfvEBtir0.jpg,05-11-2021,Sooryavanshi,FALSE,5.8,133.0,"['Action', 'Crime', 'Thriller']","['police', 'sequel', 'police officer', 'cop un...","['Akshay Kumar', 'Katrina Kaif', 'Ajay Devgn',...","['Akshay Kumar', 'Karan Johar', 'Bunty Nagi']",2021.0,5.609262,Rohit Shetty
2,FALSE,/sP9mRWiCxCuy17tUJfV8TpSZpqc.jpg,864692,Hindi,पठान,A soldier caught by enemies and presumed dead ...,47.611,/m1b9toKYyCujHuLoXB5GSDunO9e.jpg,25-01-2023,Pathaan,FALSE,6.7,70.0,"['Action', 'Adventure', 'Thriller']","['spy', 'fake death', 'spy thriller', 'spy uni...","['Shah Rukh Khan', 'Deepika Padukone', 'John A...","['Aditya Chopra', 'Siddharth Anand', 'Siddhart...",2023.0,5.628545,Siddharth Anand
3,FALSE,/vBmmJYv5asJpdJZsyPPc4MMpfBe.jpg,1018228,Hindi,चोर निकल के भागा,A flight attendant and her boyfriend must stea...,43.789,/1MIDERaEUMw1rYDM99tGZPY80Ap.jpg,24-03-2023,Chor Nikal Ke Bhaga,FALSE,7.2,55.0,"['Crime', 'Thriller']","['heist', 'airplane hijacking']","['Yami Gautam', 'Sunny Kaushal', 'Sharad Kelka...","['Dinesh Vijan', 'Raj Kumar Gupta', 'Ajay Singh']",2023.0,5.632892,Ajay Singh
4,FALSE,/u7kuUaySqXBVAtqEl9vkTkAzHV9.jpg,20453,Hindi,3 Idiots,Rascal. Joker. Dreamer. Genius... You've never...,37.260,/66A9MqXOyVFCssoloscw79z8Tew.jpg,25-12-2009,3 Idiots,FALSE,8.0,2052.0,"['Drama', 'Comedy']","['suicide', 'suicide attempt', 'college', 'mus...","['Aamir Khan', 'R. Madhavan', 'Sharman Joshi',...","['Boman Irani', 'Kareena Kapoor Khan', 'Manish...",2009.0,6.664261,Rajkumar Hirani


In [3]:
def parse_list(x):
    if pd.isna(x): return []
    if isinstance(x, list): return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return [i.strip() for i in x.split(",")]
    return [str(x)]

In [4]:
def safe_year(val):
    try:
        return str(pd.to_datetime(val, errors="coerce").year)
    except:
        return ""

In [5]:
def generate_description(row):
    title = row["title"]
    year = safe_year(row["release_date"])
    genres = ", ".join(parse_list(row["genres"]))
    cast = ", ".join(parse_list(row["cast"])[:3])
    director = row["director"]
    overview = row["overview"] or ""
    rating = row["vote_average"]

    return (
        f"{title} ({year}) is a {genres} movie directed by {director}. "
        f"It stars {cast}. Overview: {overview}. "
        f"Rating: {rating}/10. Genres: {genres}."
    )

In [6]:
df["description"] = df.apply(generate_description, axis=1)
df.to_csv("../data/RAG_cleaned_data.csv", index=False)

print("Cleaned dataset generated.")

C:\Users\PRATHAMESH\AppData\Local\Temp\ipykernel_22852\3345996914.py:3: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return str(pd.to_datetime(val, errors="coerce").year)


Cleaned dataset generated.


In [7]:
# Preview
df["description"][158]

'Hindi Medium (2017) is a Comedy, Drama movie directed by Saket Chaudhary. It stars Irrfan Khan, Saba Qamar, Sanjana Sanghi. Overview: Mita and Raj Batra, an affluent couple from Delhi’s Chandni Chowk, are grappling with getting their daughter admission into an English medium school. But there is one big problem. Their zubaan is Hindi, and the elitist snobs won’t let the Hindi speaking hoi-polloi fit in.. Rating: 7.3/10. Genres: Comedy, Drama.'

In [10]:
from uuid import uuid4
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid

In [15]:
import sys
!{sys.executable} -m pip install langchain

Defaulting to user installation because normal site-packages is not writeable


In [16]:
import sys
!{sys.executable} -m pip install langchain-text-splitters

Defaulting to user installation because normal site-packages is not writeable


In [9]:
import sys
!{sys.executable} -m pip install chromadb

Defaulting to user installation because normal site-packages is not writeable
  Using cached posthog-5.4.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
  Using cached backoff-2.2.1-py3-none-any.whl.metadata (14 kB)
  Using cached pyproject_hooks-1.2.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached durationpy-0.10-py3-none-any.whl.metadata (340 bytes)
  Using cached coloredlogs-15.0.1-py2.py3-none-any.whl.metadata (12 kB)
  Using cached humanfriendly-10.0-py2.py3-none-any.whl.metadata (9.2 kB)
  Using cached pyreadline3-3.5.4-py3-none-any.whl.metadata (4.7 kB)
  Using cached oauthlib-3.3.1-py3-none-any.whl.metadata (7.9 kB)
   ---------------------------------------- 0.0/21.4 MB ? eta -:--:--
   ----- ---------------------------------- 3.1/21.4 MB 16.4 MB/s eta 0:00:02
   ---------- ----------------------------- 5.8/21.4 MB 14.1 MB/s eta 0:00:

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

   ------------------------------------ --- 27/30 [onnxruntime]
   ------------------------------------ --- 27/30 [onnxruntime]
   ------------------------------------ --- 27/30 [onnxruntime]
   ------------------------------------ --- 27/30 [onnxruntime]
   ------------------------------------ --- 27/30 [onnxruntime]
   ------------------------------------ --- 27/30 [onnxruntime]
   ------------------------------------ --- 27/30 [onnxruntime]
   ------------------------------------ --- 27/30 [onnxruntime]
   ------------------------------------ --- 27/30 [onnxruntime]
   ------------------------------------ --- 27/30 [onnxruntime]
   ------------------------------------ --- 27/30 [onnxruntime]
   ------------------------------------ --- 27/30 [onnxruntime]
   ------------------------------------ --- 27/30 [onnxruntime]
   ------------------------------------ --- 27/30 [onnxruntime]
   ------------------------------------ --- 27/30 [onnxruntime]
   ------------------------------------ 

In [ ]:
# df = df[df["movie_index"] != 15835]

In [11]:
df = df.reset_index().rename(columns={"index": "movie_index"})

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=40)

documents, ids, metadatas = [], [], []

df["movie_index"] = df.index

In [12]:
print(df.columns.tolist())

['movie_index', 'adult', 'backdrop_path', 'movie_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'release_date', 'title', 'video', 'vote_average', 'vote_count', 'genres', 'keywords', 'cast', 'crew', 'release_year', 'score', 'director', 'description']


In [ ]:
#df = df[df["movie_index"] != 15835]

In [13]:
for _, row in df.iterrows():
    text = str(row.get("description", ""))  # safe
    chunks = splitter.split_text(text)

    for i, ch in enumerate(chunks):
        doc_id = f"{row['movie_index']}_{i}_{uuid.uuid4().hex}"

        documents.append(ch)
        ids.append(doc_id)

        # --- SAFE METADATA EXTRACTION ---
        movie_index = int(row.get("movie_index", -1))
        title = str(row.get("title", ""))

        genres = row.get("genres", [])
        if isinstance(genres, float):   # NaN
            genres = []
        if isinstance(genres, str):     # string list → convert
            try:
                import ast
                genres = ast.literal_eval(genres)
            except:
                genres = [genres]   # fallback to list of string
                
        genres_str = ", ".join(genres)

        year = str(row.get("release_year", ""))

        movie_id = str(row.get("movie_id", ""))  # ALWAYS safe

        metadatas.append({
            "movie_index": movie_index,
            "title": title,
            "genres": genres_str,
            "year": year,
            "movie_id": movie_id
        })

In [14]:
print("len(documents) =", len(documents))
print("len(ids) =", len(ids))
print("len(metadatas) =", len(metadatas))

len(documents) = 34735
len(ids) = 34735
len(metadatas) = 34735


In [ ]:
# Do not run it df = df[df["movie_index"] != 15835] this row has special symbol, utf -8 pattern.

In [ ]:
for i in range(len(documents)):
    if i >= len(metadatas):
        print("Metadata missing at chunk index:", i)
        break

In [ ]:
bad_doc = documents[i]
print("Bad document:", bad_doc)

# show metadata we expected but missing
# The chunk belongs to a row in df. Let's retrieve it.

# Find the movie_index from the doc_id
bad_id = ids[i]
print("Bad id:", bad_id)

In [ ]:
movie_index = int(bad_id.split("_")[0])
print("Movie index causing issue:", movie_index)

In [ ]:
bad_row = df.iloc[movie_index]
print(bad_row)

In [ ]:
df.shape

In [ ]:
df = df[df["movie_index"] != 15835]

In [ ]:
df.shape

In [ ]:
# splitter = RecursiveCharacterTextSplitter(
#     chunk_size=300,
#     chunk_overlap=40,
#     length_function=len
# )

In [ ]:
# from uuid import uuid4

# documents = []
# metadatas = []
# ids = []

# for idx, row in df.iterrows():   # <-- idx = unique movie id
#     chunks = splitter.split_text(row["description"])

#     for i, chunk in enumerate(chunks):
#         # --- UNIQUE ID (100% guaranteed no duplicates) ---
#         unique_id = f"{idx}_{i}_{uuid4().hex}"

#         documents.append(chunk)

#         metadatas.append({
#             "movie_id": idx,               
#             "title": row["title"]
#         })

#         ids.append(unique_id)


In [15]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(documents)

In [16]:
np.save("../artifacts/rag_embeddings.npy", embeddings)

In [17]:
client = chromadb.PersistentClient(path="../artifacts/chroma_movies")

collection = client.get_or_create_collection(
    name="movies",
    metadata={"hnsw:space": "cosine"} 
)

In [18]:
collection = client.create_collection("movies_rag")

MAX_BATCH = 5000  

total = len(documents)
print("Total chunks:", total)

for start in range(0, total, MAX_BATCH):
    end = min(start + MAX_BATCH, total)

    batch_docs = documents[start:end]
    batch_metas = metadatas[start:end]
    batch_ids = ids[start:end]
    batch_embs = embeddings[start:end].tolist()

    collection.add(
        documents=batch_docs,
        metadatas=batch_metas,
        ids=batch_ids,
        embeddings=batch_embs
    )

    print(f"Added batch {start} → {end} ({end-start} records)")


Total chunks: 34735
Added batch 0 → 5000 (5000 records)
Added batch 5000 → 10000 (5000 records)
Added batch 10000 → 15000 (5000 records)
Added batch 15000 → 20000 (5000 records)
Added batch 20000 → 25000 (5000 records)
Added batch 25000 → 30000 (5000 records)
Added batch 30000 → 34735 (4735 records)


In [19]:
q = "movies like 3 Idiots about college"
q_emb = embed_model.encode([q], convert_to_numpy=True)

res = collection.query(query_embeddings=q_emb.tolist(), n_results=5)

# print(res["documents"][0][0])
for i in range(5):
    print("Rank", i+1)
    print("Title:", res["metadatas"][0][i]["title"])
    print("Genres:", res["metadatas"][0][i]["genres"])
    print("Year:", res["metadatas"][0][i]["year"])
    print("Distance:", res["distances"][0][i])
    print("Chunk:", res["documents"][0][i][:])
    print("\n---\n")


Rank 1
Title: The Freshman
Genres: Comedy, Crime
Year: 1990.0
Distance: 0.8203970789909363
Chunk: The Freshman (1990) is a Comedy, Crime movie directed by Andrew Bergman. It stars Marlon Brando, Matthew Broderick, Bruno Kirby. Overview: After a film student gets his belongings stolen, he meets a mobster bearing a startling resemblance to a certain cinematic godfather. Soon, he finds himself

---

Rank 2
Title: 22 Jump Street
Genres: Crime, Comedy, Action
Year: 2014.0
Distance: 0.857427716255188
Chunk: to figure out if they can have a mature relationship. If these two overgrown adolescents can grow from freshmen into real men, college might be the best thing that ever happened to them.. Rating: 6.8/10. Genres: Crime, Comedy, Action.

---

Rank 3
Title: Road Trip
Genres: Comedy, Adventure
Year: 2000.0
Distance: 0.8581191301345825
Chunk: college student films his one-night stand with a beautiful sorority girl, he discovers one of his friends has accidentally mailed the homemade porn tape 

In [ ]:
# from langchain.prompts import PromptTemplate
# from langchain.llms import Ollama
# from langchain.chains import LLMChain


In [ ]:
# def retrieve(query, k=5):
#     q_emb = embed_model.encode([query]).tolist()
#     res = collection.query(query_embeddings=q_emb, n_results=k)
#     return res


In [ ]:
# llm = Ollama(model="llama3.1:8b",temperature=0.7) 

# prompt = PromptTemplate(
#     input_variables=["query", "context"],
#     template="""
# You are a movie expert. Use ONLY the following movie data:

# {context}

# User Query: {query}

# Give an accurate, short, trustworthy answer.
# """
# )


In [ ]:
# def rag_answer(query):
#     results = retrieve(query)
#     context = "\n\n".join(results["documents"][0])

#     chain = LLMChain(llm=llm, prompt=prompt)
#     return chain.run({"query": query, "context": context})

In [ ]:
# print(rag_answer("Movies like 3 Idiots about college life"))

In [ ]:
# print(rag_answer("Compare Sooryavanshi and Pathaan"))

In [ ]:
# print(rag_answer("suggest 3 Salman khan action movies release after 2020"))